# 🍏 Apple Generative Imagery Systems - Studio (Native OpenEXR)
### 3D Spatial LookDev Pipeline (Direct Depth.exr + Normal.exr + Color ID.exr + Custom LoRA)
---
이 주피터 노트북은 Houdini의 **32-bit Depth.exr, Normal.exr, Id.exr 원본 파일을 변환 없이 ComfyUI에서 실시간 직접 로딩**하는 100% Native EXR 클라우드 스튜디오입니다.

In [ ]:
# 1. GPU 사양 확인
!nvidia-smi

In [ ]:
# 2. Google Drive 마운트 (내 LoRA 모델 및 3D EXR 가이드 연동)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 3. ComfyUI 최신 엔진 + OpenEXR + RGBA 4채널 방어형 Native EXR 노드 자동 설치
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || true
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q opencv-python-headless openexr

# 📦 [핵심] 32-bit EXR 실시간 직접 로더 커스텀 노드 생성
with open("/content/ComfyUI/custom_nodes/load_exr.py", "w") as f:
    f.write('''
import os, glob, torch, numpy as np
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
import cv2

class LoadNativeEXR:
    @classmethod
    def INPUT_TYPES(s):
        return {
            "required": {
                "folder_path": ("STRING", {"default": "/content/drive/MyDrive/Generative-Imagery-Systems/apple_lora_project/3d_guides/0000", "multiline": False}),
                "exr_file_name": ("STRING", {"default": "Depth.exr", "multiline": False}),
                "pass_type": (["Depth", "Normal", "Color_ID", "Direct_RGB"], {"default": "Depth"}),
            }
        }

    RETURN_TYPES = ("IMAGE",)
    RETURN_NAMES = ("IMAGE",)
    FUNCTION = "load_and_process_exr"
    CATEGORY = "3D_VFX_Pipeline"

    def load_and_process_exr(self, folder_path, exr_file_name, pass_type):
        target_dir = folder_path.strip()
        if not os.path.exists(target_dir):
            base_search = os.path.basename(target_dir) if os.path.basename(target_dir) else "0000"
            found_folders = glob.glob(f"/content/drive/MyDrive/**/{base_search}", recursive=True)
            if found_folders:
                target_dir = found_folders[0]
                print(f"🔍 [LoadNativeEXR] 자동 추적된 폴더: {target_dir}")
            else:
                raise FileNotFoundError(f"❌ 폴더를 찾을 수 없습니다: {folder_path}")

        target_file = exr_file_name.strip()
        full_path = os.path.join(target_dir, target_file)
        if not os.path.exists(full_path):
            all_exrs = [f for f in os.listdir(target_dir) if f.endswith('.exr') or f.endswith('.png')]
            matched = [f for f in all_exrs if pass_type.lower() in f.lower() or target_file.lower() in f.lower()]
            if matched:
                full_path = os.path.join(target_dir, matched[0])
                print(f"🎯 [LoadNativeEXR] 자동 매칭된 파일: {matched[0]}")
            elif all_exrs:
                full_path = os.path.join(target_dir, all_exrs[0])
            else:
                raise FileNotFoundError(f"❌ {target_dir} 안에 {pass_type} EXR 파일이 없습니다.")

        print(f"📦 [LoadNativeEXR] 32-bit EXR 실시간 직접 로딩 중: {full_path} ({pass_type})")
        img = cv2.imread(full_path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise ValueError(f"❌ EXR 파일 읽기 실패: {full_path}")

        # 1) Depth 정밀 정규화 (부동소수점 Z거리 -> 0~1 반전)
        if pass_type == "Depth":
            d_val = img[:, :, 0] if len(img.shape) == 3 else img
            valid = (d_val > 0.0001) & (d_val < 10000.0) & (~np.isnan(d_val)) & (~np.isinf(d_val))
            if np.any(valid):
                d_min = np.percentile(d_val[valid], 0.5)
                d_max = np.percentile(d_val[valid], 99.5)
                norm_d = np.clip((d_val - d_min) / (d_max - d_min + 1e-6), 0.0, 1.0)
                d_norm = (1.0 - norm_d)
                d_norm[~valid] = 0.0
            else:
                d_norm = np.zeros_like(d_val, dtype=np.float32)
            rgb = np.stack([d_norm, d_norm, d_norm], axis=-1)

        # 2) Normal BGR->RGB 및 [-1, 1] 디코딩
        elif pass_type == "Normal":
            if img.dtype == np.uint8:
                rgb = img.astype(np.float32) / 255.0
            else:
                rgb = np.clip((img + 1.0) * 0.5, 0.0, 1.0)
            if len(rgb.shape) == 3 and rgb.shape[-1] >= 3:
                rgb = rgb[:, :, [2, 1, 0]]

        # 3) Color ID 디코딩
        else:
            if img.dtype == np.uint8:
                rgb = img.astype(np.float32) / 255.0
            else:
                rgb = np.clip(img, 0.0, 1.0)
            if len(rgb.shape) == 3 and rgb.shape[-1] >= 3:
                rgb = rgb[:, :, [2, 1, 0]]

        # 🛡️ [핵심 방어] RGBA 4채널 -> 무조건 3채널(RGB)로 강제 고정 (KSampler 에러 원천 차단!)
        if len(rgb.shape) == 2:
            rgb = np.stack([rgb, rgb, rgb], axis=-1)
        elif len(rgb.shape) == 3:
            if rgb.shape[-1] == 1:
                rgb = np.concatenate([rgb, rgb, rgb], axis=-1)
            elif rgb.shape[-1] > 3:
                rgb = rgb[:, :, :3]

        # 🛡️ [핵심 방어] 해상도를 16:9 정밀 1024x576으로 고정
        h, w = rgb.shape[:2]
        if w != 1024 or h != 576:
            interp = cv2.INTER_NEAREST if pass_type == "Color_ID" else cv2.INTER_AREA
            rgb = cv2.resize(rgb, (1024, 576), interpolation=interp)

        tensor = torch.from_numpy(np.ascontiguousarray(rgb, dtype=np.float32)).unsqueeze(0)
        print(f"✅ [LoadNativeEXR] {pass_type} 패스 텐서 검증 통과: Shape={tensor.shape}")
        return (tensor,)

NODE_CLASS_MAPPINGS = {"LoadNativeEXR": LoadNativeEXR}
NODE_DISPLAY_NAME_MAPPINGS = {"LoadNativeEXR": "📦 Load 3D EXR Pass (Native)"}
''')

print('✅ ComfyUI 엔진 및 완벽 방어형 Native OpenEXR 노드 장착 완료!')

In [ ]:
# 4. SDXL Base + VAE + ControlNet (Depth & Normal) + LoRA 가중치 연동
import glob, os
print('🚀 필수 AI 모델 가중치 초고속 다운로드 중...')
# 1) SDXL Base 1.0 (6.4GB)
!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P /content/ComfyUI/models/checkpoints/

# 2) SDXL VAE Fix (335MB)
!wget -c https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors -P /content/ComfyUI/models/vae/

# 3) SDXL ControlNet Depth (2.5GB)
!wget -c https://huggingface.co/diffusers/controlnet-depth-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors -O /content/ComfyUI/models/controlnet/controlnet-depth-sdxl-1.0.safetensors

# 4) SDXL ControlNet Normal / Union (2.3GB)
!wget -c https://huggingface.co/xinsir/controlnet-union-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors -O /content/ComfyUI/models/controlnet/controlnet-normal-sdxl-1.0.safetensors

# 5) LoRA 가중치 자동 연동
lora_target_dir = "/content/ComfyUI/models/loras"
os.makedirs(lora_target_dir, exist_ok=True)
lora_found = glob.glob("/content/drive/MyDrive/**/apple_minimal_craft_sdxl_v1.safetensors", recursive=True)
if lora_found:
    src_lora = lora_found[0]
    dst_lora = os.path.join(lora_target_dir, "apple_minimal_craft_sdxl_v1.safetensors")
    if not os.path.exists(dst_lora):
        os.symlink(src_lora, dst_lora)
    print(f'✅ LoRA 연동 완료: {src_lora}')

print('✅ 모든 모델 준비 완료!')

In [ ]:
# 5. 🌐 ComfyUI 실행 및 Localtunnel 연결
!pkill -f "main.py" 2>/dev/null || true
!pkill -f "lt" 2>/dev/null || true
!npm install -g localtunnel > /dev/null 2>&1

import subprocess, threading, time, urllib.request

def run_comfy():
    !python /content/ComfyUI/main.py --listen 127.0.0.1 --port 8188 --highvram --dont-upcast-attention

threading.Thread(target=run_comfy, daemon=True).start()
time.sleep(5)

try:
    public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
except:
    public_ip = '136.85.6.54'

print('='*70)
print(f'🔑 Localtunnel 비밀번호(Password): {public_ip}')
print('='*70)

!npx localtunnel --port 8188
